# Z_2_2_Formatting_Macro_Economics_Data



**Objective:**  
To convert raw macroeconomic files into clean, analysis-ready `.parquet` datasets for use in predictive modeling and feature engineering.

**Datasets Processed:**
- Annual Personal Income and Employment (APIE)
- Personal Consumption Expenditures (PCE)
- Quarterly Personal Income (QPI)
- Quarterly GDP (QGDP)

**Key Steps:**
- Removed metadata/footnotes and dropped irrelevant years/columns
- Reshaped wide-to-long-to-wide format using `melt` and `pivot_table`
- Converted string-based years/quarters to datetime format
- Saved cleaned datasets in efficient `.parquet` format

**Output Files:**
- `Annual Personal Income and Employment.parquet`
- `Personal consumption expenditures.parquet`
- `Quarterly personal income.parquet`
- `Quarterly GDP.parquet`


In [7]:
# ===========================================
# 1. Import Required Libraries and Settings
# ===========================================
import pandas as pd
import re
import pyarrow as pa

# Disable warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')


In [8]:
# ===========================================
# 2. Define File Paths for Raw and Formatted Data
# ===========================================

# Path to raw macroeconomic files for 2008
raw_files_path = r"data\Macroeconomic data\Raw Files\2008"

# Path to save cleaned/structured outputs
formatted_path = r"data\Macroeconomic data\Formatted\2008"

# Note: All cleaned datasets will be saved in .parquet format for storage efficiency


In [9]:
# ===========================================
# 3. Process Annual Personal Income and Employment (APIE)
# ===========================================

# Load APIE CSV file while skipping the first 3 non-data rows
apie = pd.read_csv(rf"{raw_files_path}\Annual personal income and employment.csv", skiprows=3)

# Preview top rows
apie.head(5)

# Inspect bottom rows (may contain footnotes or totals)
apie.tail(15)

# Drop last 13 rows that are footnotes or metadata
apie = apie.iloc[:-13].reset_index(drop=True)

# Drop non-essential columns and years not needed
apie.drop(columns=["GeoFips", "LineCode", "2008", "2024"], inplace=True)

# Identify all year columns (e.g., "2000", "2010", etc.)
year_columns = [col for col in apie.columns if str(col).isdigit()]

# Reshape from wide to long format
apie_long = apie.melt(id_vars=["GeoName", "Description"], 
                      value_vars=year_columns,
                      var_name="Year", 
                      value_name="Value")

# Convert string-based values to numeric (e.g., "(NA)" → NaN)
apie_long['Value'] = pd.to_numeric(apie_long['Value'], errors='coerce')

# Pivot back to wide format, creating one column per economic metric
apie_wide = apie_long.pivot_table(
    index=["GeoName", "Year"],
    columns="Description",
    values="Value",
    aggfunc='first'  # Avoid aggregation error for text columns
).reset_index()

# Convert 'Year' to datetime (end of year)
apie_wide["Year"] = pd.to_datetime(apie_wide["Year"], errors="coerce") + pd.offsets.YearEnd(0)
apie_wide = apie_wide.dropna(subset=["Year"])  # Drop invalid dates

# Save cleaned file
apie_wide.to_parquet(rf"{formatted_path}\Annual Personal Income and Employment.parquet")


In [10]:
# ===========================================
# 4. Process Personal Consumption Expenditures (PCE)
# ===========================================

# Check for malformed lines in the CSV
file_path = rf"{raw_files_path}\Personal consumption expenditures.csv"
min_expected_columns = 5

with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        columns = line.strip().split(",")
        if len(columns) < min_expected_columns:
            print(f"Bad line at index {i} (only {len(columns)} columns):")
            print(repr(line.strip()))

# Load data skipping metadata
pce = pd.read_csv(file_path, skiprows=3).reset_index()

# Drop last 3 rows and unnecessary columns
pce = pce.iloc[:-3].reset_index(drop=True)
pce.drop(columns=["GeoFips", "LineCode", "index"], inplace=True)

# Identify year columns
year_columns = [col for col in pce.columns if str(col).isdigit()]

# Reshape from wide to long format
pce_long = pce.melt(id_vars=["GeoName", "Description"], 
                    value_vars=year_columns,
                    var_name="Year", 
                    value_name="Value")

# Convert Value to numeric and pivot back to wide
pce_long['Value'] = pd.to_numeric(pce_long['Value'], errors='coerce')
pce_wide = pce_long.pivot_table(
    index=["GeoName", "Year"],
    columns="Description",
    values="Value",
    aggfunc='first'
).reset_index()

# Convert Year to end-of-year datetime
pce_wide["Year"] = pd.to_datetime(pce_wide["Year"], errors="coerce") + pd.offsets.YearEnd(0)
pce_wide = pce_wide.dropna(subset=["Year"])

# Save cleaned file
pce_wide.to_parquet(rf"{formatted_path}\Personal consumption expenditures.parquet")


Bad line at index 0 (only 1 columns):
'SAPCE1 Personal consumption expenditures (PCE) by major type of product 1'
Bad line at index 1 (only 1 columns):
'SAPCE1 Personal consumption expenditures (PCE) by major type of product 1'
Bad line at index 2 (only 1 columns):
'State or DC'
Bad line at index 1444 (only 1 columns):
''
Bad line at index 1445 (only 1 columns):
'Legend/Footnotes'
Bad line at index 1446 (only 1 columns):
'1. All personal consumption expenditures (PCE) estimates are in millions of current dollars (not adjusted for inflation). Calculations are performed on unrounded data.'
Bad line at index 1447 (only 2 columns):
'"  Last updated: October 3, 2024-- new statistics for 2023; revised statistics for 2019-2022."'


In [11]:
# ===========================================
# 5. Process Quarterly Personal Income (QPI)
# ===========================================

# Check for malformed lines
file_path = rf"{raw_files_path}\Quarterly personal income.csv"

with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        columns = line.strip().split(",")
        if len(columns) < min_expected_columns:
            print(f"Bad line at index {i} (only {len(columns)} columns):")
            print(repr(line.strip()))

# Load QPI CSV file
qpi = pd.read_csv(file_path, skiprows=3)

# Drop last 7 rows which likely contain footnotes
qpi = qpi.iloc[:-7].reset_index(drop=True)

# Drop unused columns and specific 2008 quarterly columns
qpi.drop(columns=["GeoFips", "2008:Q1", "2008:Q2", "2008:Q3", "LineCode"], inplace=True)

# Clean up geographic names by removing footnote symbols or special characters
qpi["GeoName"] = qpi["GeoName"].str.replace(r"[^\w\s]+$", "", regex=True).str.strip()

# The reshaping and export steps will follow next


Bad line at index 0 (only 3 columns):
'"SQINC1 State quarterly personal income summary: personal income, population, per capita personal income"'
Bad line at index 1 (only 3 columns):
'"SQINC1 State quarterly personal income summary: personal income, population, per capita personal income"'
Bad line at index 2 (only 1 columns):
'State or DC'
Bad line at index 184 (only 1 columns):
''
Bad line at index 185 (only 1 columns):
'Legend/Footnotes'
Bad line at index 186 (only 1 columns):
'1. Midquarter population estimates by state are derived by BEA based on unpublished U.S. Census Bureau estimates of beginning-of-month population. Quarterly estimates for 2020-2024 reflect unpublished monthly population estimates available as of February 2025.'
Bad line at index 187 (only 1 columns):
'2. Per capita personal income is total personal income divided by total quarterly population estimates.'
Bad line at index 188 (only 1 columns):
'* Estimates prior to 1950 are not available for Alaska and Hawai

In [12]:
# ===========================================
# 6. Process Quarterly Personal Income (QPI) - Continued
# ===========================================

# Helper function to extract quarterly column names (e.g., "2008:Q4")
def get_quarter_columns(qpi):
    pattern = re.compile(r"^\d{4}:Q[1-4]$")
    return [col for col in qpi.columns if pattern.match(str(col))]

# Identify quarterly columns for reshaping
year_columns = get_quarter_columns(qpi)

# Melt quarterly data into long format
qpi_long = qpi.melt(id_vars=["GeoName", "Description"], 
                    value_vars=year_columns,
                    var_name="Year", 
                    value_name="Value")

# Convert string values to numeric (coerce '(NA)' to NaN)
qpi_long['Value'] = pd.to_numeric(qpi_long['Value'], errors='coerce')

# Pivot to wide format with Description as columns
qpi_wide = qpi_long.pivot_table(
    index=["GeoName", "Year"],
    columns="Description",
    values="Value",
    aggfunc='first'
).reset_index()

# Convert quarter to end-of-quarter datetime
def convert_quarter_to_end_date(q):
    year, quarter = q.split(":")
    month = {"Q1": "03", "Q2": "06", "Q3": "09", "Q4": "12"}[quarter]
    return pd.Timestamp(f"{year}-{month}-01") + pd.offsets.MonthEnd(0)

qpi_wide["Year"] = qpi_wide["Year"].apply(convert_quarter_to_end_date)

# Preview transformed dataset
qpi_wide

# Save formatted QPI data
qpi_wide.to_parquet(rf"{formatted_path}\Quarterly personal income.parquet")


In [13]:
# ===========================================
# 7. Process Quarterly GDP (QGDP)
# ===========================================

# Check for malformed lines in the raw QGDP file
file_path = rf"{raw_files_path}\Quarterly GDP.csv"
min_expected_columns = 5

with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        columns = line.strip().split(",")
        if len(columns) < min_expected_columns:
            print(f"Bad line at index {i} (only {len(columns)} columns):")
            print(repr(line.strip()))

# Load QGDP CSV, skipping metadata rows
qgdp = pd.read_csv(file_path, skiprows=3)

# Drop footnotes/summary rows at the bottom
qgdp = qgdp.iloc[:-3].reset_index(drop=True)

# Drop non-relevant columns including unneeded quarters and metadata
qgdp.drop(columns=["GeoFips", "2008:Q1", "2008:Q2", "2008:Q3", "LineCode"], inplace=True)

# Clean geographic names by stripping footnotes and extra characters
qgdp["GeoName"] = qgdp["GeoName"].str.replace(r"[^\w\s]+$", "", regex=True).str.strip()

# Identify quarterly columns in QGDP
def get_quarter_columns(qgdp):
    pattern = re.compile(r"^\d{4}:Q[1-4]$")
    return [col for col in qgdp.columns if pattern.match(str(col))]

year_columns = get_quarter_columns(qgdp)

# Melt long
qgdp_long = qgdp.melt(id_vars=["GeoName", "Description"], 
                      value_vars=year_columns,
                      var_name="Year", 
                      value_name="Value")

# Convert values to numeric
qgdp_long['Value'] = pd.to_numeric(qgdp_long['Value'], errors='coerce')

# Pivot to wide format
qgdp_wide = qgdp_long.pivot_table(
    index=["GeoName", "Year"],
    columns="Description",
    values="Value",
    aggfunc='first'
).reset_index()

# Convert Year (quarter string) to end-of-quarter datetime
def convert_quarter_to_end_date(q):
    year, quarter = q.split(":")
    month = {"Q1": "03", "Q2": "06", "Q3": "09", "Q4": "12"}[quarter]
    return pd.Timestamp(f"{year}-{month}-01") + pd.offsets.MonthEnd(0)

qgdp_wide["Year"] = qgdp_wide["Year"].apply(convert_quarter_to_end_date)

# Preview the result
qgdp_wide

# Save cleaned QGDP dataset
qgdp_wide.to_parquet(rf"{formatted_path}\Quarterly GDP.parquet")


Bad line at index 0 (only 1 columns):
'SQGDP1 State quarterly gross domestic product (GDP) summary'
Bad line at index 1 (only 1 columns):
'SQGDP1 State quarterly gross domestic product (GDP) summary'
Bad line at index 2 (only 1 columns):
'State or DC'
Bad line at index 184 (only 1 columns):
''
Bad line at index 185 (only 1 columns):
'Legend/Footnotes'
Bad line at index 187 (only 2 columns):
'"  Last updated: March 28, 2025-- new statistics for 2024:Q4."'


# END